# Navitas — Model Eğitimi (Colab / GPU) — Drive'SIZ sürüm

**Google Drive GEREKMEZ** (premium ödemen gerekmiyor). Veriler Colab'ın kendi diskine
iner, eğitilen model sonunda **GitHub'a push edilir** — bilgisayarında `git pull` ile alırsın.

**Nasıl:** Üstte **Çalışma Zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç.
Sonra her hücreyi sırayla **Shift+Enter** ile çalıştır.

Model: `yolov8n-seg`, 2 sınıf (`traversable`/`blocked`) · Veri: **RescueNet + FloodNet** · 100 epoch


## Adım 1 — GPU var mı?
Çıktıda bir GPU (ör. Tesla T4) görmelisin.

In [ ]:
!nvidia-smi

## Adım 2 — Projeyi GitHub'dan indir
Çalıştırınca **GitHub token** isteyecek (görünmez). Bu token hem indirmek hem de
sonunda modeli geri push etmek için kullanılır — `navitas26/navitas`'a erişimi olmalı.

In [ ]:
from getpass import getpass
GITHUB_TOKEN = getpass("GitHub token yapıştır (görünmez): ")
REPO = "navitas26/navitas"
!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git
!ls navitas

## Adım 3 — Gerekli kütüphaneleri kur

In [ ]:
!pip install ultralytics datasets pillow tqdm opencv-python-headless -q
print('Kurulum tamam')

## Adım 4 — Veri setlerini indir: RescueNet + FloodNet
Drive YOK; veriler Colab diskine iner (~10-20 dk). İki veri seti de iner.

In [ ]:
%cd /content/navitas/dataset
print("=== RescueNet indiriliyor ===")
!python3 download_and_convert.py
print("=== FloodNet indiriliyor ===")
!python3 download_floodnet.py
print("Veri setleri hazır")

## Adım 5 — Eğitim ayar dosyasını yaz (HER İKİ veri seti dahil)

In [ ]:
yaml_content = """train:
  - /content/navitas/dataset/rescuenet_yolo/images/train
  - /content/navitas/dataset/floodnet_yolo/images/train

val:
  - /content/navitas/dataset/rescuenet_yolo/images/val
  - /content/navitas/dataset/floodnet_yolo/images/val

test: /content/navitas/dataset/rescuenet_yolo/images/test

nc: 2
names:
  0: traversable
  1: blocked
"""
with open("/content/navitas/dataset/combined_data.yaml","w") as f:
    f.write(yaml_content)
print(yaml_content)
print("Ayar yazildi (RescueNet + FloodNet)")

## Adım 6 — GPU için eğitim ayarı (device=0, batch=16)

In [ ]:
!sed -i 's/device="cpu"/device=0/' /content/navitas/codes/train_seg.py
!sed -i 's/batch=8,/batch=16,/' /content/navitas/codes/train_seg.py
!grep -nE "device=|batch=" /content/navitas/codes/train_seg.py

## Adım 7 — Eğit (100 epoch · ~1-2 saat)
Uzun sürer; sekmeyi kapatma.

In [ ]:
%cd /content/navitas/codes
!python3 train_seg.py --data /content/navitas/dataset/combined_data.yaml --epochs 100

## Adım 8 — Eğitilmiş modeli GitHub'a PUSH et
Model dosyalarını repoya ekleyip `navitas26/navitas`'a gönderir.
Bilgisayarında `git pull` yapınca model gelir. (Drive'a gerek yok.)

In [ ]:
import glob, shutil, os
os.chdir('/content/navitas')
hits = sorted(glob.glob('codes/runs/segment/**/weights/best.pt', recursive=True))
print('Bulunan best.pt:', hits)
os.makedirs('models', exist_ok=True)
shutil.copy(hits[0], 'models/navitas_seg_v2_best.pt')
print('best.pt -> models/navitas_seg_v2_best.pt')

!git config user.email "hattab1342@gmail.com"
!git config user.name "navitas-colab"
!git add -f models/navitas_seg_v2_best.pt
!git add -f models/navitas_seg_v2_ncnn_model
!git commit -m "model: navitas_seg_v2 (Colab egitim, RescueNet+FloodNet, 100ep)"
!git pull --no-edit origin master
!git push origin HEAD:master
print(">>> Bitti. Hata yoksa model GitHub'da. Bilgisayarinda: git pull")

## Adım 9 (YEDEK) — Push olmazsa modeli doğrudan indir
Adım 8'de push hata verirse bunu çalıştır; model zip olarak bilgisayarına iner.

In [ ]:
import shutil
from google.colab import files
shutil.copytree('/content/navitas/models/navitas_seg_v2_ncnn_model',
                '/content/cikti/navitas_seg_v2_ncnn_model', dirs_exist_ok=True)
shutil.copy('/content/navitas/models/navitas_seg_v2_best.pt',
            '/content/cikti/navitas_seg_v2_best.pt')
shutil.make_archive('/content/navitas_model', 'zip', '/content/cikti')
files.download('/content/navitas_model.zip')